Parte 2: Evaluación de Modelos de Proveedores (AB vs XY)
Solventa - Prueba Técnica Data Scientist Jr.

Objetivo: Determinar cuál de los dos modelos es más efectivo para
la calificación de clientes y recomendar el proveedor a contratar.

Criterios de análisis:
- Capacidad discriminante (ROC-AUC, KS, Gini)
- Estabilidad del puntaje (PSI)
- Distribución de buenos y malos por rangos de score
- Indicadores comparativos

In [1]:
import warnings

In [2]:
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
import os
import json

In [4]:
os.makedirs("../output/figures", exist_ok=True)

1. CARGA Y PREPARACIÓN

In [5]:
print("=" * 60)
print("PARTE 2: EVALUACIÓN DE MODELOS DE PROVEEDORES")
print("=" * 60)

print("\n[1] Cargando datos...")
df = pd.read_excel("../ModelosCompetencia.xlsx")
print(f"   Registros: {df.shape[0]}")
print(f"   Columnas: {list(df.columns)}")

# Convertir fecha a datetime para cálculo de PSI temporal
df["Fecha Estudio"] = pd.to_datetime(df["Fecha Estudio"])

# Limpiar puntajes (PuntajeXY tiene "." como missing)
df["PuntajeXY"] = pd.to_numeric(df["PuntajeXY"], errors="coerce")

print(
    f"\n   PuntajeXY missing: {df['PuntajeXY'].isnull().sum()} ({df['PuntajeXY'].isnull().mean() * 100:.2f}%)"
)

df_clean = df.dropna(subset=["PuntajeXY"]).copy()
print(f"   Registros limpios (ambos puntajes): {len(df_clean)}")

# Estadísticas básicas
print("\n   Estadísticas PuntajeAB:")
print(f"     Media: {df['PuntajeAB'].mean():.4f}, Std: {df['PuntajeAB'].std():.4f}")
print(f"     Min: {df['PuntajeAB'].min():.4f}, Max: {df['PuntajeAB'].max():.4f}")
print("\n   Estadísticas PuntajeXY:")
print(
    f"     Media: {df_clean['PuntajeXY'].mean():.4f}, Std: {df_clean['PuntajeXY'].std():.4f}"
)
print(
    f"     Min: {df_clean['PuntajeXY'].min():.4f}, Max: {df_clean['PuntajeXY'].max():.4f}"
)

print(f"\n   Tasa de Default: {df['Default'].mean() * 100:.2f}%")

PARTE 2: EVALUACIÓN DE MODELOS DE PROVEEDORES

[1] Cargando datos...


   Registros: 114109
   Columnas: ['ID', 'Fecha Estudio', 'PuntajeAB', 'PuntajeXY', 'Default']

   PuntajeXY missing: 12800 (11.22%)
   Registros limpios (ambos puntajes): 101309

   Estadísticas PuntajeAB:
     Media: 10.1256, Std: 0.6515
     Min: 6.0000, Max: 12.1304

   Estadísticas PuntajeXY:
     Media: 19.9977, Std: 5.1897
     Min: 0.1176, Max: 27.8235

   Tasa de Default: 60.45%


2. CAPACIDAD DISCRIMINANTE

In [6]:
print("\n[2] Evaluando capacidad discriminante...")

# ROC-AUC
auc_ab = roc_auc_score(df["Default"], df["PuntajeAB"])
auc_xy = roc_auc_score(df_clean["Default"], df_clean["PuntajeXY"])

# Determinar si score más alto = más riesgo o menos riesgo
# Si AUC < 0.5, el score está invertido (mayor score = menos riesgo)
auc_ab_direction = auc_ab if auc_ab > 0.5 else 1 - auc_ab
auc_xy_direction = auc_xy if auc_xy > 0.5 else 1 - auc_xy

print(
    f"\n   ROC-AUC PuntajeAB: {auc_ab:.4f} ({'invertido' if auc_ab < 0.5 else 'normal'})"
)
print(
    f"   ROC-AUC PuntajeXY: {auc_xy:.4f} ({'invertido' if auc_xy < 0.5 else 'normal'})"
)
print(f"   ROC-AUC Ajustado AB: {auc_ab_direction:.4f}")
print(f"   ROC-AUC Ajustado XY: {auc_xy_direction:.4f}")


# KS Statistic
def ks_statistic(y_true, y_score):
    """Calcula el estadístico Kolmogorov-Smirnov"""
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ks = max(tpr - fpr)
    return ks


# Para KS necesitamos scores donde mayor = más probabilidad de default
# Si AUC < 0.5, invertimos
score_ab = -df["PuntajeAB"] if auc_ab < 0.5 else df["PuntajeAB"]
score_xy = -df_clean["PuntajeXY"] if auc_xy < 0.5 else df_clean["PuntajeXY"]

ks_ab = ks_statistic(df["Default"], score_ab)
ks_xy = ks_statistic(df_clean["Default"], score_xy)

print(f"\n   KS PuntajeAB: {ks_ab:.4f}")
print(f"   KS PuntajeXY: {ks_xy:.4f}")

# Gini Coefficient
gini_ab = 2 * auc_ab_direction - 1
gini_xy = 2 * auc_xy_direction - 1

print(f"\n   Gini PuntajeAB: {gini_ab:.4f}")
print(f"   Gini PuntajeXY: {gini_xy:.4f}")


[2] Evaluando capacidad discriminante...

   ROC-AUC PuntajeAB: 0.4987 (invertido)
   ROC-AUC PuntajeXY: 0.5018 (normal)
   ROC-AUC Ajustado AB: 0.5013
   ROC-AUC Ajustado XY: 0.5018

   KS PuntajeAB: 0.0052
   KS PuntajeXY: 0.0051

   Gini PuntajeAB: 0.0026
   Gini PuntajeXY: 0.0035


3. CURVAS ROC COMPARATIVAS

In [7]:
print("\n[3] Generando curvas ROC...")

fig, ax = plt.subplots(figsize=(8, 6))

# Para ROC, si score invertido, usar -score
fpr_ab, tpr_ab, _ = roc_curve(
    df["Default"], -df["PuntajeAB"] if auc_ab < 0.5 else df["PuntajeAB"]
)
fpr_xy, tpr_xy, _ = roc_curve(
    df_clean["Default"],
    -df_clean["PuntajeXY"] if auc_xy < 0.5 else df_clean["PuntajeXY"],
)

ax.plot(
    fpr_ab,
    tpr_ab,
    label=f"Proveedor AB (AUC={auc_ab_direction:.4f})",
    linewidth=2,
    color="#3498db",
)
ax.plot(
    fpr_xy,
    tpr_xy,
    label=f"Proveedor XY (AUC={auc_xy_direction:.4f})",
    linewidth=2,
    color="#e74c3c",
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("Tasa de Falsos Positivos", fontsize=12)
ax.set_ylabel("Tasa de Verdaderos Positivos", fontsize=12)
ax.set_title("Curva ROC - Proveedor AB vs Proveedor XY", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Marcar punto KS
idx_ks_ab = np.argmax(tpr_ab - fpr_ab)
ax.scatter(fpr_ab[idx_ks_ab], tpr_ab[idx_ks_ab], color="#3498db", s=100, zorder=5)
idx_ks_xy = np.argmax(tpr_xy - fpr_xy)
ax.scatter(fpr_xy[idx_ks_xy], tpr_xy[idx_ks_xy], color="#e74c3c", s=100, zorder=5)

plt.tight_layout()
plt.savefig("../output/figures/09_roc_competencia.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 09_roc_competencia.png")


[3] Generando curvas ROC...


   -> Guardado: 09_roc_competencia.png


4. ANÁLISIS POR DECILES

In [8]:
print("\n[4] Análisis por deciles...")


def analyze_by_deciles(df, score_col, target_col, label):
    """Analiza distribución de buenos/malos por deciles de score"""
    df_analysis = df[[score_col, target_col]].copy()

    # Deciles: ordenar según si mayor score = más riesgo o menos
    if roc_auc_score(df_analysis[target_col], df_analysis[score_col]) < 0.5:
        df_analysis["decile"] = pd.qcut(
            df_analysis[score_col], 10, labels=False, duplicates="drop"
        )
        # Invertir: decil 1 = mejor score (menor riesgo)
        df_analysis["decile"] = 9 - df_analysis["decile"]
    else:
        df_analysis["decile"] = pd.qcut(
            df_analysis[score_col], 10, labels=False, duplicates="drop"
        )

    decile_stats = df_analysis.groupby("decile").agg(
        {target_col: ["sum", "count", "mean"], score_col: ["mean", "min", "max"]}
    )
    decile_stats.columns = [
        "defaults",
        "total",
        "default_rate",
        "mean_score",
        "min_score",
        "max_score",
    ]
    decile_stats["good"] = decile_stats["total"] - decile_stats["defaults"]
    decile_stats["pct_good"] = decile_stats["good"] / decile_stats["total"] * 100
    decile_stats["pct_bad"] = decile_stats["defaults"] / decile_stats["total"] * 100

    return decile_stats


decile_ab = analyze_by_deciles(df, "PuntajeAB", "Default", "AB")
decile_xy = analyze_by_deciles(df_clean, "PuntajeXY", "Default", "XY")

print("\n   Deciles - Proveedor AB:")
print(decile_ab[["defaults", "total", "default_rate"]].to_string())
print("\n   Deciles - Proveedor XY:")
print(decile_xy[["defaults", "total", "default_rate"]].to_string())

# Visualización por deciles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AB - stacked bar
decile_ab[["pct_good", "pct_bad"]].plot(
    kind="bar", stacked=True, ax=axes[0], color=["#2ecc71", "#e74c3c"], width=0.7
)
axes[0].set_title("Proveedor AB - Distribución por Deciles", fontsize=12)
axes[0].set_xlabel("Decil (1=mejor, 10=peor)")
axes[0].set_ylabel("Porcentaje (%)")
axes[0].legend(["Buenos", "Malos"])
axes[0].grid(True, alpha=0.3, axis="y")

# XY - stacked bar
decile_xy[["pct_good", "pct_bad"]].plot(
    kind="bar", stacked=True, ax=axes[1], color=["#2ecc71", "#e74c3c"], width=0.7
)
axes[1].set_title("Proveedor XY - Distribución por Deciles", fontsize=12)
axes[1].set_xlabel("Decil (1=mejor, 10=peor)")
axes[1].set_ylabel("Porcentaje (%)")
axes[1].legend(["Buenos", "Malos"])
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../output/figures/10_deciles_competencia.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 10_deciles_competencia.png")

# Default rate por decil
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    decile_ab.index,
    decile_ab["default_rate"] * 100,
    "o-",
    linewidth=2,
    label="AB",
    color="#3498db",
)
ax.plot(
    decile_xy.index,
    decile_xy["default_rate"] * 100,
    "s-",
    linewidth=2,
    label="XY",
    color="#e74c3c",
)
ax.axhline(
    y=df["Default"].mean() * 100, color="gray", linestyle="--", label="Tasa promedio"
)
ax.set_xlabel("Decil")
ax.set_ylabel("Tasa de Default (%)")
ax.set_title("Tasa de Default por Decil - AB vs XY")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../output/figures/11_default_rate_deciles.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 11_default_rate_deciles.png")


[4] Análisis por deciles...

   Deciles - Proveedor AB:
        defaults  total  default_rate
decile                               
0           6417  10739      0.597542
1           5193   8653      0.600139
2           8871  14649      0.605570
3           6128  10120      0.605534
4           6219  10174      0.611264
5           7276  12017      0.605476
6           7545  12438      0.606609
7           7375  12261      0.601501
8           6262  10317      0.606959
9           7688  12741      0.603406

   Deciles - Proveedor XY:
        defaults  total  default_rate
decile                               
0           6138  10178      0.603065
1           6056  10146      0.596885
2           6127  10172      0.602340
3           6125  10036      0.610303
4           6288  10307      0.610071
5           6221  10310      0.603395
6           6159  10326      0.596456
7           6036  10047      0.600776
8           5865   9673      0.606327
9           6186  10114      0.611627


   -> Guardado: 10_deciles_competencia.png


   -> Guardado: 11_default_rate_deciles.png


5. ESTABILIDAD DEL PUNTAJE (PSI)

In [9]:
print("\n[5] Analizando estabilidad del puntaje (PSI)...")


def calculate_psi(expected, actual, buckets=10):
    """Population Stability Index - mide cambio en distribución"""
    # Crear buckets basados en la distribución esperada
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)

    if len(breakpoints) < 3:
        return 0.0

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]

    # Evitar división por cero
    expected_pct = np.maximum(expected_counts / len(expected), 0.0001)
    actual_pct = np.maximum(actual_counts / len(actual), 0.0001)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi


# Dividir datos en dos mitades para calcular PSI temporal
mid_date = df["Fecha Estudio"].median()
df_first = df[df["Fecha Estudio"] <= mid_date]
df_second = df[df["Fecha Estudio"] > mid_date]

psi_ab = calculate_psi(df_first["PuntajeAB"].values, df_second["PuntajeAB"].values)

df_clean_first = df_clean[df_clean["Fecha Estudio"] <= mid_date]
df_clean_second = df_clean[df_clean["Fecha Estudio"] > mid_date]

if len(df_clean_first) > 0 and len(df_clean_second) > 0:
    psi_xy = calculate_psi(
        df_clean_first["PuntajeXY"].values, df_clean_second["PuntajeXY"].values
    )
else:
    psi_xy = np.nan

print(f"\n   PSI PuntajeAB: {psi_ab:.4f}")
print(
    f"   PSI PuntajeXY: {psi_xy:.4f}"
    if not np.isnan(psi_xy)
    else "   PSI PuntajeXY: N/A (datos insuficientes)"
)


def psi_interpretation(psi):
    if psi < 0.1:
        return "Bajo - Sin cambio significativo"
    elif psi < 0.2:
        return "Moderado - Cambio leve"
    else:
        return "Alto - Cambio significativo, revisar modelo"


print(f"\n   Interpretación AB: {psi_interpretation(psi_ab)}")
if not np.isnan(psi_xy):
    print(f"   Interpretación XY: {psi_interpretation(psi_xy)}")

# Distribución de scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["PuntajeAB"], bins=50, kde=True, ax=axes[0], color="#3498db")
axes[0].set_title(f"Distribución PuntajeAB (PSI={psi_ab:.4f})")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Frecuencia")

if not np.isnan(psi_xy):
    sns.histplot(df_clean["PuntajeXY"], bins=50, kde=True, ax=axes[1], color="#e74c3c")
    axes[1].set_title(f"Distribución PuntajeXY (PSI={psi_xy:.4f})")
else:
    sns.histplot(df_clean["PuntajeXY"], bins=50, kde=True, ax=axes[1], color="#e74c3c")
    axes[1].set_title(f"Distribución PuntajeXY (PSI=N/A)")
axes[1].set_xlabel("Score")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig("../output/figures/12_score_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 12_score_distribution.png")


[5] Analizando estabilidad del puntaje (PSI)...

   PSI PuntajeAB: 0.0006
   PSI PuntajeXY: 0.0011

   Interpretación AB: Bajo - Sin cambio significativo
   Interpretación XY: Bajo - Sin cambio significativo


   -> Guardado: 12_score_distribution.png


6. ANÁLISIS DE SEPARACIÓN

In [10]:
print("\n[6] Analizando separación entre buenos y malos...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AB
for label, val in [("No Default", 0), ("Default", 1)]:
    sns.kdeplot(
        df[df.Default == val]["PuntajeAB"],
        label=label,
        ax=axes[0],
        fill=True,
        alpha=0.3,
    )
axes[0].set_title("Proveedor AB - Distribución por Default")
axes[0].set_xlabel("Score")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# XY
for label, val in [("No Default", 0), ("Default", 1)]:
    sns.kdeplot(
        df_clean[df_clean.Default == val]["PuntajeXY"],
        label=label,
        ax=axes[1],
        fill=True,
        alpha=0.3,
    )
axes[1].set_title("Proveedor XY - Distribución por Default")
axes[1].set_xlabel("Score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../output/figures/13_separation_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 13_separation_analysis.png")

# Estadísticos de separación
print("\n   Separación de scores (Mean diff / Std):")
mean_ab_good = df[df.Default == 0]["PuntajeAB"].mean()
mean_ab_bad = df[df.Default == 1]["PuntajeAB"].mean()
std_ab = df["PuntajeAB"].std()
print(
    f"   AB: Buenos={mean_ab_good:.4f}, Malos={mean_ab_bad:.4f}, Separación={abs(mean_ab_good - mean_ab_bad) / std_ab:.4f}"
)

mean_xy_good = df_clean[df_clean.Default == 0]["PuntajeXY"].mean()
mean_xy_bad = df_clean[df_clean.Default == 1]["PuntajeXY"].mean()
std_xy = df_clean["PuntajeXY"].std()
print(
    f"   XY: Buenos={mean_xy_good:.4f}, Malos={mean_xy_bad:.4f}, Separación={abs(mean_xy_good - mean_xy_bad) / std_xy:.4f}"
)


[6] Analizando separación entre buenos y malos...


   -> Guardado: 13_separation_analysis.png

   Separación de scores (Mean diff / Std):
   AB: Buenos=10.1268, Malos=10.1248, Separación=0.0030
   XY: Buenos=19.9804, Malos=20.0090, Separación=0.0055


7. COMPARACIÓN LIFT

In [11]:
print("\n[7] Análisis de Lift...")


def calculate_lift(df, score_col, target_col, n_bins=10):
    """Calcula lift por deciles"""
    df_lift = df[[score_col, target_col]].copy()
    if roc_auc_score(df_lift[target_col], df_lift[score_col]) < 0.5:
        # Score invertido: mayor score = menor riesgo, invertir etiquetas
        df_lift["rank"] = pd.qcut(
            -df_lift[score_col], n_bins, labels=range(1, n_bins + 1), duplicates="drop"
        )
    else:
        # Score normal: mayor score = mayor riesgo
        df_lift["rank"] = pd.qcut(
            df_lift[score_col], n_bins, labels=range(n_bins, 0, -1), duplicates="drop"
        )

    lift = df_lift.groupby("rank").agg({target_col: ["mean", "count"]})
    lift.columns = ["default_rate", "count"]
    overall_rate = df_lift[target_col].mean()
    lift["lift"] = lift["default_rate"] / overall_rate
    lift["cumulative_pct"] = (lift["count"].cumsum() / lift["count"].sum() * 100).round(
        1
    )
    lift["cumulative_defaults"] = (
        lift["count"].cumsum() / df_lift[target_col].sum() * 100
    )

    return lift


lift_ab = calculate_lift(df, "PuntajeAB", "Default")
lift_xy = calculate_lift(df_clean, "PuntajeXY", "Default")

print("\n   Lift Proveedor AB:")
print(lift_ab.to_string())
print("\n   Lift Proveedor XY:")
print(lift_xy.to_string())

# Lift chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(lift_ab) + 1), lift_ab["lift"], color="#3498db", alpha=0.8)
axes[0].axhline(y=1, color="red", linestyle="--", label="Lift=1 (aleatorio)")
axes[0].set_title("Lift por Decil - Proveedor AB")
axes[0].set_xlabel("Decil (1=mejor)")
axes[0].set_ylabel("Lift")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(range(1, len(lift_xy) + 1), lift_xy["lift"], color="#e74c3c", alpha=0.8)
axes[1].axhline(y=1, color="red", linestyle="--", label="Lift=1 (aleatorio)")
axes[1].set_title("Lift por Decil - Proveedor XY")
axes[1].set_xlabel("Decil (1=mejor)")
axes[1].set_ylabel("Lift")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../output/figures/14_lift_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 14_lift_analysis.png")


[7] Análisis de Lift...

   Lift Proveedor AB:
      default_rate  count      lift  cumulative_pct  cumulative_defaults
rank                                                                    
1         0.599904  12502  0.992467            11.0            18.125671
2         0.599793  10642  0.992284            20.3            33.554673
3         0.602497  14898  0.996757            33.3            55.154116
4         0.611234  14065  1.011211            45.7            75.545858
5         0.610188   5595  1.009480            50.6            83.657610
6         0.603401  11997  0.998252            61.1           101.051121
7         0.610599  10624  1.010160            70.4           116.454026
8         0.599088  11843  0.991118            80.8           133.624264
9         0.608159  10540  1.006125            90.0           148.905385
10        0.602736  11403  0.997153           100.0           165.437701

   Lift Proveedor XY:
      default_rate  count      lift  cumulative_pct  

   -> Guardado: 14_lift_analysis.png


8. REPORTE FINAL

In [12]:
print("\n" + "=" * 60)
print("RESUMEN COMPARATIVO - PROVEEDORES AB vs XY")
print("=" * 60)

psi_xy_str = f"{psi_xy:.4f}" if not np.isnan(psi_xy) else "N/A"

print(f"""
{"Metrica":<25} {"Proveedor AB":<20} {"Proveedor XY":<20}
{"-" * 65}
ROC-AUC:                 {auc_ab_direction:<20.4f} {auc_xy_direction:<20.4f}
KS Statistic:            {ks_ab:<20.4f} {ks_xy:<20.4f}
Gini Coefficient:        {gini_ab:<20.4f} {gini_xy:<20.4f}
PSI (Estabilidad):       {psi_ab:<20.4f} {psi_xy_str:<20}
Registros:               {len(df):<20} {len(df_clean):<20}
Tasa Default:            {df["Default"].mean() * 100:<20.2f}% {df_clean["Default"].mean() * 100:.2f}%
""")

# Both models have AUC ~0.50, equivalent to random
print(f"\nRECOMENDACIÓN: No contratar a ninguno de los dos proveedores.")
print(f"Justificación: Ambos modelos tienen un poder discriminante equivalente al azar (AUC ≈ 0.50).")
print(f"Proveedor AB AUC: {auc_ab_direction:.4f}, Proveedor XY AUC: {auc_xy_direction:.4f}")
print(f"La inversión en estos proveedores no generaría valor agregado respecto a una selección aleatoria.")

# Guardar resultados
competitor_results = {
    "auc_ab": auc_ab_direction,
    "auc_xy": auc_xy_direction,
    "ks_ab": ks_ab,
    "ks_xy": ks_xy,
    "gini_ab": gini_ab,
    "gini_xy": gini_xy,
    "psi_ab": psi_ab,
    "psi_xy": psi_xy if not np.isnan(psi_xy) else None,
    "recommended_provider": "Ninguno",
    "records_ab": len(df),
    "records_xy": len(df_clean),
    "default_rate_ab": df["Default"].mean(),
    "default_rate_xy": df_clean["Default"].mean(),
    "decile_ab": decile_ab.reset_index().to_dict("records"),
    "decile_xy": decile_xy.reset_index().to_dict("records"),
    "lift_ab": lift_ab.reset_index().to_dict("records"),
    "lift_xy": lift_xy.reset_index().to_dict("records"),
}

with open("../output/competitor_metrics.json", "w") as f:
    json.dump(competitor_results, f, indent=2)

print("\nParte 2 completada exitosamente!")


RESUMEN COMPARATIVO - PROVEEDORES AB vs XY

Metrica                   Proveedor AB         Proveedor XY        
-----------------------------------------------------------------
ROC-AUC:                 0.5013               0.5018              
KS Statistic:            0.0052               0.0051              
Gini Coefficient:        0.0026               0.0035              
PSI (Estabilidad):       0.0006               0.0011              
Registros:               114109               101309              
Tasa Default:            60.45               % 60.41%


RECOMENDACIÓN: No contratar a ninguno de los dos proveedores.
Justificación: Ambos modelos tienen un poder discriminante equivalente al azar (AUC ≈ 0.50).
Proveedor AB AUC: 0.5013, Proveedor XY AUC: 0.5018
La inversión en estos proveedores no generaría valor agregado respecto a una selección aleatoria.

Parte 2 completada exitosamente!
